In [ ]:
import pandas as pd
import re
from aloud_database.aloud_database import Database

db = Database()

In [ ]:
query_buyers_emails = """
    SELECT
        c.lead_id,
        e.email,
        p.formatted_phone,
        p.whatsapp_format
    FROM
        lead_tracking_prod.conversions c 
    LEFT JOIN   
        lead_tracking_prod.emails e 
            ON e.id = c.email_id
    LEFT JOIN   
        lead_tracking_prod.phone p 
            ON p.id = c.phone_id
    WHERE
        c.conversion_type_id = '8'
"""
buyers_data = db.execute_query(query=query_buyers_emails)

# Checagem se o DataFrame está vazio antes de manipular colunas
if not buyers_data.empty:
    # Remover espaços extras, normalizar para string e tratar valores nulos
    buyers_data['email'] = buyers_data['email'].astype(str).str.lower().str.strip()
    buyers_data['formatted_phone'] = buyers_data['formatted_phone'].astype(str).str.replace("+", "", case=False, regex=False).str.strip()
    buyers_data['whatsapp_format'] = buyers_data['whatsapp_format'].astype(str).str.replace("+", "", case=False, regex=False).str.strip()

    # Remover entradas vazias, nulas ou 'nan'
    buyers_data['email'] = buyers_data['email'].replace(['', 'nan', 'None', 'NaN'], pd.NA).dropna()
    buyers_data['formatted_phone'] = buyers_data['formatted_phone'].replace(['', 'nan', 'None', 'NaN'], pd.NA).dropna()
    buyers_data['whatsapp_format'] = buyers_data['whatsapp_format'].replace(['', 'nan', 'None', 'NaN'], pd.NA).dropna()

    # Coletar valores únicos já limpos e não nulos
    buyer_emails = buyers_data['email'][buyers_data['email'].notna()].unique()
    buyer_fphones = buyers_data['formatted_phone'][buyers_data['formatted_phone'].notna()].unique()
    buyer_wphones = buyers_data['whatsapp_format'][buyers_data['whatsapp_format'].notna()].unique()
else:
    buyer_emails = []
    buyer_fphones = []
    buyer_wphones = []

In [ ]:
query_inscritos_bf = """
    SELECT
        c.lead_id,
        e.email,
        p.formatted_phone,
        p.whatsapp_format
    FROM
        lead_tracking_prod.conversions c 
    LEFT JOIN   
        lead_tracking_prod.emails e 
            ON e.id = c.email_id
    LEFT JOIN   
        lead_tracking_prod.phone p 
            ON p.id = c.phone_id
    WHERE
        c.conversion_type_id = '1'
        and c.campaign_id like 'lcto-bf25%'
"""

bf_leads = db.execute_query(query=query_inscritos_bf)

# Checagem se o DataFrame está vazio antes de manipular colunas
if not buyers_data.empty:
    # Remover espaços extras, normalizar para string e tratar valores nulos
    bf_leads['email'] = bf_leads['email'].astype(str).str.lower().str.strip()
    bf_leads['formatted_phone'] = bf_leads['formatted_phone'].astype(str).str.replace("+", "", case=False, regex=False).str.strip()
    bf_leads['whatsapp_format'] = bf_leads['whatsapp_format'].astype(str).str.replace("+", "", case=False, regex=False).str.strip()

    # Remover entradas vazias, nulas ou 'nan'
    bf_leads['email'] = bf_leads['email'].replace(['', 'nan', 'None', 'NaN'], pd.NA).dropna()
    bf_leads['formatted_phone'] = bf_leads['formatted_phone'].replace(['', 'nan', 'None', 'NaN'], pd.NA).dropna()
    bf_leads['whatsapp_format'] = bf_leads['whatsapp_format'].replace(['', 'nan', 'None', 'NaN'], pd.NA).dropna()

    # Coletar valores únicos já limpos e não nulos
    b25_emails = bf_leads['email'][bf_leads['email'].notna()].unique()
    b25_fphones = bf_leads['formatted_phone'][bf_leads['formatted_phone'].notna()].unique()
    b25_wphones = bf_leads['whatsapp_format'][bf_leads['whatsapp_format'].notna()].unique()
else:
    b25_emails = []
    b25_fphones = []
    b25_fphones = []

In [ ]:
emails_to_filter = set(buyer_emails).union(set(b25_emails))
phones_to_filter = set(buyer_fphones).union(set(buyer_wphones)).union(set(b25_fphones)).union(set(b25_wphones))


def filter_leads(main_df: pd.DataFrame, emails_to_filter = emails_to_filter, phones_to_filter = phones_to_filter):
    # Garante que os sets/iteráveis estão no formato certo
    emails_to_filter = set([str(e).strip().lower() for e in emails_to_filter if pd.notna(e)])
    phones_to_filter = set([str(p).strip() for p in phones_to_filter if pd.notna(p)])

    # Normaliza emails para comparação
    main_df['email_norm'] = main_df['email'].astype(str).str.strip().str.lower()
    main_df['phone_norm'] = main_df['phone'].astype(str).str.strip()

    # Filtragem direta, conforme solicitado
    filtered_df = main_df[~main_df['email_norm'].isin(emails_to_filter)].copy()
    
    # Filtragem direta, conforme solicitado
    filtered_df = main_df[~main_df['phone_norm'].isin(phones_to_filter)].copy()

    # Remove colunas auxiliares
    filtered_df.drop(columns=['email_norm', 'phone_norm'], inplace=True)

    return filtered_df

In [ ]:
query_template = f"""
SELECT 
    c.lead_id,
    l.name,
    l.profile_score,
    e.email,
    p.formatted_phone as phone,
    c.conversion_raw_info
FROM 
    lead_tracking_prod.conversions c
JOIN 
    lead_tracking_prod.leads l ON c.lead_id = l.id
JOIN 
    lead_tracking_prod.emails e ON c.email_id = e.id
JOIN 
    lead_tracking_prod.phone p ON c.phone_id = p.id
WHERE 
    c.conversion_type_id = 2
"""

df = db.execute_query(query = query_template )

# Lista de variações possíveis
income_keys = [
    "monthly_income",
    "monthly_incomme",
    "monthly_personal_income",
    "monthy_income"
]

# Função para extrair o valor de income do JSON
def get_value(row, keys):
    if not isinstance(row, dict):
        return None
    for k in keys:
        if k in row:
            return row[k]
    return None

# Criando as novas colunas
df["monthly_incomme"] = df["conversion_raw_info"].apply(lambda x: get_value(x, income_keys))
df["current_occupation"] = df["conversion_raw_info"].apply(lambda x: x.get('current_occupation'))


In [ ]:
campaign_to_filter = ['BF24', 'OFAN_25', 'OFAN_25R', 'OFAN_25RMET', 'SDI25', 'lcto-dsi-set/25', 'isca-aulassemanais']


for campaign_id in campaign_to_filter:
    def get_campaign_leads(campaign_id: str):
        query_template = f"""
        SELECT 
            c.lead_id,
            l.name,
            e.email,
            p.formatted_phone as phone,
            l.profile_score,
            c.campaign_id
        FROM 
            lead_tracking_prod.conversions c
        JOIN 
            lead_tracking_prod.leads l ON c.lead_id = l.id
        JOIN 
            lead_tracking_prod.emails e ON c.lead_id = e.lead_id
        JOIN 
            lead_tracking_prod.phone p ON c.lead_id = p.lead_id
        WHERE 
            c.conversion_type_id = 1
            AND c.campaign_id = '{campaign_id}'
            AND NOT EXISTS (
                SELECT 1 
                FROM lead_tracking_prod.conversions cx 
                WHERE cx.lead_id = c.lead_id 
                AND cx.conversion_type_id = 8
            )
            AND NOT EXISTS (
                SELECT 1 
                FROM lead_tracking_prod.conversions cy 
                WHERE cy.lead_id = c.lead_id 
                AND cy.conversion_type_id = 1 
                AND cy.campaign_id ILIKE 'lcto-bf25%'
            );
        """

        return db.execute_query(query=query_template)

    def clean_data(df):
        df['phone'] = df['phone'].apply(lambda x: re.sub(r'[^0-9]', '', str(x)))
        df['phone'] = df['phone'].replace(r'^\s*$', pd.NA, regex=True)
        return df

    df = get_campaign_leads(campaign_id)

    df = clean_data(df)
    filtered_leads = filter_leads(df)

    filtered_leads.to_csv(f'export/leads_{campaign_id.replace("_","-").replace("/","").lower()}.csv')


